In [1]:
import pandas as pd
import json
import pickle
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print("Libraries imported!")

Libraries imported!


In [2]:
chunks_df = pd.read_csv('data/chunks_512.csv')

with open('data/test_questions.json', 'r') as f:
    test_questions = json.load(f)

corpus = chunks_df['text'].tolist()

print(f"Loaded {len(chunks_df)} chunks")
print(f"Loaded {len(test_questions)} test questions")

Loaded 3725 chunks
Loaded 20 test questions


In [3]:
print("Loading embedding model...")
print("First time takes 2-3 minutes to download...")

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded!")

Loading embedding model...
First time takes 2-3 minutes to download...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\HI\Desktop\rag-arxiv-research\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HI\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!


In [4]:
print("Encoding all chunks into vectors...")
print("This takes 5-10 minutes on your system...")

embeddings = model.encode(
    corpus,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")
print("Encoding complete!")

Encoding all chunks into vectors...
This takes 5-10 minutes on your system...


Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Embeddings shape: (3725, 384)
Encoding complete!


In [5]:
print("Building FAISS index...")

dimension = embeddings.shape[1]
print(f"Embedding dimension: {dimension}")

# Build flat L2 index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"FAISS index built!")
print(f"Total vectors in index: {index.ntotal}")

Building FAISS index...
Embedding dimension: 384
FAISS index built!
Total vectors in index: 3725


In [6]:
os.makedirs('src/retrievers', exist_ok=True)

# Save FAISS index
faiss.write_index(index, 'src/retrievers/faiss_index.bin')

# Save embeddings
np.save('src/retrievers/embeddings.npy', embeddings)

print("FAISS index saved!")

FAISS index saved!


In [7]:
def dense_retrieve(query, top_k=5):
    # Encode query
    query_vector = model.encode([query], convert_to_numpy=True)
    
    # Search FAISS index
    distances, indices = index.search(
        query_vector.astype('float32'), top_k
    )
    
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'chunk_id': chunks_df.iloc[idx]['chunk_id'],
            'text': corpus[idx],
            'distance': distances[0][i]
        })
    
    return results

print("Dense retriever ready!")

Dense retriever ready!


In [8]:
print("=== TESTING DENSE RETRIEVER ===\n")

for i, item in enumerate(test_questions[:5]):
    query = item['question']
    results = dense_retrieve(query, top_k=3)
    
    print(f"Question {i+1}: {query}")
    print(f"Top result distance: {results[0]['distance']:.4f}")
    print(f"Top result preview: {results[0]['text'][:150]}...")
    print("-" * 50)

=== TESTING DENSE RETRIEVER ===

Question 1: What type of system is being analyzed in the paper for the mean resolvent using a polymer expansion?
Top result distance: 0.9641
Top result preview: in this paper we develop a polymer expansion with large / small field conditions for the mean resolvent of a weakly disordered system . then we show t...
--------------------------------------------------
Question 2: What is the significance of the asymptotic expansion for the density of states in the context of the research paper?
Top result distance: 0.9803
Top result preview: in this paper we develop a polymer expansion with large / small field conditions for the mean resolvent of a weakly disordered system . then we show t...
--------------------------------------------------
Question 3: What is the asymptotic long-time equivalence being referred to in the context of the paper?
Top result distance: 1.0639
Top result preview: we show the asymptotic long - time equivalence of a generic power l

In [9]:
print("Evaluating Dense Retriever on all test questions...\n")

dense_results = []

for item in tqdm(test_questions):
    query = item['question']
    results = dense_retrieve(query, top_k=5)
    
    source = item['source_abstract']
    retrieved_texts = [r['text'] for r in results]
    
    hit = any(
        len(set(source.lower().split()) & 
            set(r.lower().split())) > 20 
        for r in retrieved_texts
    )
    
    dense_results.append({
        'question': query,
        'hit': hit,
        'top_distance': results[0]['distance'],
        'top_result': results[0]['text']
    })

hits = sum(1 for r in dense_results if r['hit'])
accuracy = hits / len(dense_results) * 100

print(f"Dense Retriever Results:")
print(f"Total questions: {len(dense_results)}")
print(f"Correct retrievals: {hits}")
print(f"Accuracy: {accuracy:.1f}%")

Evaluating Dense Retriever on all test questions...



100%|██████████| 20/20 [00:00<00:00, 27.13it/s]

Dense Retriever Results:
Total questions: 20
Correct retrievals: 20
Accuracy: 100.0%


In [11]:
# Convert numpy types to Python native types
dense_results_clean = []
for r in dense_results:
    dense_results_clean.append({
        'question': r['question'],
        'hit': bool(r['hit']),
        'top_distance': float(r['top_distance']),
        'top_result': r['top_result']
    })

with open('results/dense_results.json', 'w') as f:
    json.dump(dense_results_clean, f, indent=2)

summary = {
    'retriever': 'Dense (FAISS)',
    'total_questions': len(dense_results),
    'hits': hits,
    'accuracy': float(accuracy)
}

with open('results/dense_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Results saved!")
print(f"\nDense Retriever Summary:")
print(f"Accuracy: {accuracy:.1f}%")

Results saved!

Dense Retriever Summary:
Accuracy: 100.0%


In [12]:
with open('results/bm25_summary.json', 'r') as f:
    bm25_summary = json.load(f)

print("=" * 40)
print("   RETRIEVER COMPARISON SO FAR")
print("=" * 40)
print(f"BM25 Accuracy:  {bm25_summary['accuracy']:.1f}%")
print(f"Dense Accuracy: {accuracy:.1f}%")
print("=" * 40)
print("\nHybrid retriever coming tomorrow!")

   RETRIEVER COMPARISON SO FAR
BM25 Accuracy:  100.0%
Dense Accuracy: 100.0%

Hybrid retriever coming tomorrow!
